# English-French Machine Translation with Encoder-Decoder LSTM

**Project:** English to French Translation using LSTM

**Objective:** Implement from scratch an Encoder-Decoder LSTM model for machine translation

## Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Data Preparation](#2-data-preparation)
3. [Model Architecture](#3-model-architecture)
4. [Training](#4-training)
5. [Evaluation](#5-evaluation)
6. [Error Analysis](#6-error-analysis)

## 1. Environment Setup

First, install all required dependencies.

In [1]:
# Install required packages
%pip install spacy torch torchtext nltk matplotlib requests
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm

Note: you may need to restart the kernel to use updated packages.
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 4.2 MB/s eta 0:00:03
     -- ------------------------------------- 0.8/12.8 MB 2.6 MB/s eta 0:00:05
     --- ------------------------------------ 1.0/12.8 MB 2.3 MB/s eta 0:00:06
     ---- ----------------------------------- 1.3/12.8 MB 1.8 MB/s eta 0:00:07
     ---- ----------------------------------- 1.3/12.8 MB 1.8 MB/s eta 0:00:07
     ---- ----------------------------------- 1.3/12.8 MB 1.8 MB/s eta 0:00:07
     ---- ----------------------------------- 1.6/12.8 MB 1.1 MB/s eta 0:00:11
     ----- ---------------------------------- 1.8/12.8 MB 1.0 MB/s eta 0:00:11
     ----- ---------------------------------- 1.8/12.8 MB 1.0 MB/s eta 0:00:11
     ------ -------------------------------- 2.1/12.8 MB 987.1 kB/s eta 0:00:11
     ------- ------------------------------- 2.4/12.8 MB 979.5 kB/s eta

In [2]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

import spacy
from collections import Counter
import random
import math
import time
import requests
import os

import matplotlib.pyplot as plt
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 2. Data Preparation

### 2.1 Download Multi30K Dataset (English-French)

In [ ]:
# Download Multi30K dataset (en-fr)
# The Multi30K dataset can be downloaded using torchtext or manually

import urllib.request
import gzip
import shutil

def download_and_extract(url, output_file):
    """Download file and extract if gzipped"""
    try:
        # Download file
        print(f'Downloading {output_file}...')
        urllib.request.urlretrieve(url, output_file + '.tmp')
        
        # Check if it's a gzip file
        try:
            with gzip.open(output_file + '.tmp', 'rb') as f_in:
                with open(output_file, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            print(f'Extracted to {output_file}')
            os.remove(output_file + '.tmp')
        except:
            # Not a gzip file, just rename
            shutil.move(output_file + '.tmp', output_file)
            print(f'Saved to {output_file}')
        
        return True
    except Exception as e:
        print(f'Error downloading {output_file}: {e}')
        if os.path.exists(output_file + '.tmp'):
            os.remove(output_file + '.tmp')
        return False

# Create data directory
os.makedirs('data', exist_ok=True)

print("Attempting to download Multi30K dataset...")
print("Note: If automatic download fails, please download manually from:")
print("https://github.com/multi30k/dataset/tree/master/data/task1/raw\n")

# Try using torchtext datasets first (most reliable method)
try:
    from torchtext.datasets import Multi30k
    from torchtext.data.utils import get_tokenizer
    
    print("Using torchtext to download Multi30K dataset...")
    
    # This will download the dataset
    # Note: Multi30k in torchtext is en-de by default, so we need alternative approach
    print("Note: torchtext Multi30K is en-de. We need en-fr from raw files.\n")
    use_torchtext = False
except:
    use_torchtext = False

# Manual download URLs (FIXED: Using task1 for English-French pairs)
# Task1 contains: English, French, German, Czech
# Task2 contains: English, German ONLY (no French!)
base_url = 'https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw/'

files_to_download = {
    'data/train.en': base_url + 'train.en.gz',
    'data/train.fr': base_url + 'train.fr.gz', 
    'data/val.en': base_url + 'val.en.gz',
    'data/val.fr': base_url + 'val.fr.gz',
    'data/test.en': base_url + 'test_2016_flickr.en.gz',
    'data/test.fr': base_url + 'test_2016_flickr.fr.gz'
}

# Check if files already exist
all_exist = all(os.path.exists(f) for f in files_to_download.keys())

if all_exist:
    print("Dataset files already exist!")
else:
    print("Downloading dataset files...")
    success_count = 0
    
    for output_file, url in files_to_download.items():
        if os.path.exists(output_file):
            print(f'{output_file} already exists, skipping...')
            success_count += 1
        else:
            if download_and_extract(url, output_file):
                success_count += 1
    
    if success_count == len(files_to_download):
        print('\n✓ Dataset downloaded and extracted successfully!')
    else:
        print(f'\n⚠ Warning: Only {success_count}/{len(files_to_download)} files downloaded.')
        print('\nIf download failed, please manually download from:')
        print('https://github.com/multi30k/dataset/tree/master/data/task1/raw')
        print('\nManual steps:')
        print('1. Click on each .gz file (train.en.gz, train.fr.gz, etc.)')
        print('2. Click "Download raw file"')
        print('3. Extract the .gz files')
        print('4. Place the extracted files in the "data" folder')
        print('\nAlternatively, you can create a small sample dataset for testing:')
        
        # Create sample data if download completely failed
        if success_count == 0:
            print('\nCreating sample dataset for testing...')
            sample_data_en = [
                "A man in a blue shirt is standing on a ladder.",
                "Two young girls are playing in the park.",
                "A dog is running on the beach.",
                "The woman is reading a book.",
                "Children are playing soccer."
            ] * 200  # Repeat to create ~1000 samples
            
            sample_data_fr = [
                "Un homme en chemise bleue est debout sur une échelle.",
                "Deux jeunes filles jouent dans le parc.",
                "Un chien court sur la plage.",
                "La femme lit un livre.",
                "Les enfants jouent au football."
            ] * 200
            
            # Write sample training data
            with open('data/train.en', 'w', encoding='utf-8') as f:
                f.write('\n'.join(sample_data_en[:800]))
            
            with open('data/train.fr', 'w', encoding='utf-8') as f:
                f.write('\n'.join(sample_data_fr[:800]))
            
            # Write sample validation data
            with open('data/val.en', 'w', encoding='utf-8') as f:
                f.write('\n'.join(sample_data_en[800:900]))
            
            with open('data/val.fr', 'w', encoding='utf-8') as f:
                f.write('\n'.join(sample_data_fr[800:900]))
            
            # Write sample test data  
            with open('data/test.en', 'w', encoding='utf-8') as f:
                f.write('\n'.join(sample_data_en[900:1000]))
            
            with open('data/test.fr', 'w', encoding='utf-8') as f:
                f.write('\n'.join(sample_data_fr[900:1000]))
            
            print('✓ Sample dataset created for testing!')
            print('Note: This is a small sample. For the actual project, please download the full Multi30K dataset.')

# Verify files exist
print("\nVerifying dataset files...")
for filepath in files_to_download.keys():
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            num_lines = sum(1 for _ in f)
        print(f'✓ {filepath}: {num_lines} sentences')
    else:
        print(f'✗ {filepath}: NOT FOUND')

### 2.2 Load and Tokenize Data

In [ ]:
# Load spaCy models for tokenization
spacy_en = spacy.load('en_core_web_sm')
spacy_fr = spacy.load('fr_core_news_sm')

def tokenize_en(text):
    """Tokenize English text using spaCy"""
    return [tok.text.lower() for tok in spacy_en.tokenizer(text)]

def tokenize_fr(text):
    """Tokenize French text using spaCy"""
    return [tok.text.lower() for tok in spacy_fr.tokenizer(text)]

# Test tokenizers
print('English tokenization:', tokenize_en('Hello, how are you?'))
print('French tokenization:', tokenize_fr('Bonjour, comment allez-vous?'))

In [ ]:
# Load dataset
def load_data(en_file, fr_file):
    """Load parallel English-French sentences"""
    with open(en_file, 'r', encoding='utf-8') as f:
        en_sentences = [line.strip() for line in f]
    
    with open(fr_file, 'r', encoding='utf-8') as f:
        fr_sentences = [line.strip() for line in f]
    
    return list(zip(en_sentences, fr_sentences))

# Load train, validation, and test sets
train_data = load_data('data/train.en', 'data/train.fr')
val_data = load_data('data/val.en', 'data/val.fr')
test_data = load_data('data/test.en', 'data/test.fr')

print(f'Train set: {len(train_data)} pairs')
print(f'Validation set: {len(val_data)} pairs')
print(f'Test set: {len(test_data)} pairs')
print(f'\nExample pair:')
print(f'EN: {train_data[0][0]}')
print(f'FR: {train_data[0][1]}')

### 2.3 Build Vocabulary

In [ ]:
class Vocabulary:
    """Vocabulary class for converting tokens to indices and vice versa"""
    
    def __init__(self, max_size=10000):
        self.token2idx = {}
        self.idx2token = {}
        self.max_size = max_size
        
        # Special tokens
        self.PAD_TOKEN = '<pad>'
        self.UNK_TOKEN = '<unk>'
        self.SOS_TOKEN = '<sos>'
        self.EOS_TOKEN = '<eos>'
        
        self.PAD_IDX = 0
        self.UNK_IDX = 1
        self.SOS_IDX = 2
        self.EOS_IDX = 3
        
        # Initialize with special tokens
        self.token2idx = {
            self.PAD_TOKEN: self.PAD_IDX,
            self.UNK_TOKEN: self.UNK_IDX,
            self.SOS_TOKEN: self.SOS_IDX,
            self.EOS_TOKEN: self.EOS_IDX
        }
        self.idx2token = {v: k for k, v in self.token2idx.items()}
    
    def build_vocab(self, tokenized_sentences):
        """Build vocabulary from tokenized sentences"""
        counter = Counter()
        for tokens in tokenized_sentences:
            counter.update(tokens)
        
        # Get most common tokens (excluding special tokens)
        most_common = counter.most_common(self.max_size - 4)
        
        # Add to vocabulary
        for token, _ in most_common:
            if token not in self.token2idx:
                idx = len(self.token2idx)
                self.token2idx[token] = idx
                self.idx2token[idx] = token
        
        print(f'Vocabulary size: {len(self.token2idx)}')
    
    def encode(self, tokens):
        """Convert tokens to indices"""
        return [self.token2idx.get(token, self.UNK_IDX) for token in tokens]
    
    def decode(self, indices):
        """Convert indices to tokens"""
        return [self.idx2token.get(idx, self.UNK_TOKEN) for idx in indices]
    
    def __len__(self):
        return len(self.token2idx)

# Build vocabularies
print('Building English vocabulary...')
en_vocab = Vocabulary(max_size=10000)
en_tokenized = [tokenize_en(en) for en, fr in train_data]
en_vocab.build_vocab(en_tokenized)

print('\nBuilding French vocabulary...')
fr_vocab = Vocabulary(max_size=10000)
fr_tokenized = [tokenize_fr(fr) for en, fr in train_data]
fr_vocab.build_vocab(fr_tokenized)

### 2.4 Create Dataset and DataLoader with Padding/Packing

In [ ]:
class TranslationDataset(Dataset):
    """Custom Dataset for translation pairs"""
    
    def __init__(self, data, en_vocab, fr_vocab, en_tokenizer, fr_tokenizer, max_len=50):
        self.data = data
        self.en_vocab = en_vocab
        self.fr_vocab = fr_vocab
        self.en_tokenizer = en_tokenizer
        self.fr_tokenizer = fr_tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        en_text, fr_text = self.data[idx]
        
        # Tokenize
        en_tokens = self.en_tokenizer(en_text)
        fr_tokens = self.fr_tokenizer(fr_text)
        
        # Limit length
        en_tokens = en_tokens[:self.max_len]
        fr_tokens = fr_tokens[:self.max_len]
        
        # Add EOS token
        en_tokens.append(self.en_vocab.EOS_TOKEN)
        fr_tokens.append(self.fr_vocab.EOS_TOKEN)
        
        # Convert to indices
        en_indices = self.en_vocab.encode(en_tokens)
        fr_indices = self.fr_vocab.encode(fr_tokens)
        
        return torch.tensor(en_indices), torch.tensor(fr_indices)

def collate_fn(batch):
    """Custom collate function for padding and packing"""
    # Separate source and target
    src_batch, tgt_batch = zip(*batch)
    
    # Get lengths
    src_lengths = torch.tensor([len(s) for s in src_batch])
    tgt_lengths = torch.tensor([len(t) for t in tgt_batch])
    
    # Sort by source length (descending) for pack_padded_sequence
    src_lengths, sort_idx = src_lengths.sort(descending=True)
    src_batch = [src_batch[i] for i in sort_idx]
    tgt_batch = [tgt_batch[i] for i in sort_idx]
    tgt_lengths = tgt_lengths[sort_idx]
    
    # Pad sequences
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    
    return src_padded, tgt_padded, src_lengths, tgt_lengths

# Create datasets
train_dataset = TranslationDataset(train_data, en_vocab, fr_vocab, tokenize_en, tokenize_fr)
val_dataset = TranslationDataset(val_data, en_vocab, fr_vocab, tokenize_en, tokenize_fr)
test_dataset = TranslationDataset(test_data, en_vocab, fr_vocab, tokenize_en, tokenize_fr)

# Create dataloaders
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Train batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

## 3. Model Architecture

### 3.1 Encoder

In [ ]:
class Encoder(nn.Module):
    """LSTM Encoder for sequence-to-sequence model"""
    
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super(Encoder, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        # Embedding layer
        self.embedding = nn.Embedding(input_dim, embedding_dim, padding_idx=0)
        
        # LSTM layer
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            n_layers,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, src, src_lengths):
        """
        Args:
            src: [batch_size, src_len] - source sequence
            src_lengths: [batch_size] - lengths of source sequences
        
        Returns:
            outputs: [batch_size, src_len, hidden_dim] - all hidden states
            hidden: [n_layers, batch_size, hidden_dim] - final hidden state
            cell: [n_layers, batch_size, hidden_dim] - final cell state
        """
        # Embed source sequence
        embedded = self.dropout(self.embedding(src))
        # embedded: [batch_size, src_len, embedding_dim]
        
        # Pack padded sequence
        packed_embedded = pack_padded_sequence(
            embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=True
        )
        
        # Pass through LSTM
        packed_outputs, (hidden, cell) = self.lstm(packed_embedded)
        
        # Unpack sequence
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True)
        # outputs: [batch_size, src_len, hidden_dim]
        # hidden: [n_layers, batch_size, hidden_dim]
        # cell: [n_layers, batch_size, hidden_dim]
        
        return outputs, hidden, cell

### 3.2 Decoder

In [ ]:
class Decoder(nn.Module):
    """LSTM Decoder for sequence-to-sequence model"""
    
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super(Decoder, self).__init__()
        
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        # Embedding layer
        self.embedding = nn.Embedding(output_dim, embedding_dim, padding_idx=0)
        
        # LSTM layer
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            n_layers,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        
        # Output layer
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input, hidden, cell):
        """
        Args:
            input: [batch_size] - current input token
            hidden: [n_layers, batch_size, hidden_dim] - previous hidden state
            cell: [n_layers, batch_size, hidden_dim] - previous cell state
        
        Returns:
            prediction: [batch_size, output_dim] - output probabilities
            hidden: [n_layers, batch_size, hidden_dim] - current hidden state
            cell: [n_layers, batch_size, hidden_dim] - current cell state
        """
        # Add sequence dimension
        input = input.unsqueeze(1)
        # input: [batch_size, 1]
        
        # Embed input
        embedded = self.dropout(self.embedding(input))
        # embedded: [batch_size, 1, embedding_dim]
        
        # Pass through LSTM
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # output: [batch_size, 1, hidden_dim]
        
        # Remove sequence dimension and pass through linear layer
        prediction = self.fc_out(output.squeeze(1))
        # prediction: [batch_size, output_dim]
        
        return prediction, hidden, cell

### 3.3 Seq2Seq Model

In [ ]:
class Seq2Seq(nn.Module):
    """Sequence-to-Sequence model combining Encoder and Decoder"""
    
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, src, src_lengths, tgt, teacher_forcing_ratio=0.5):
        """
        Args:
            src: [batch_size, src_len] - source sequence
            src_lengths: [batch_size] - lengths of source sequences
            tgt: [batch_size, tgt_len] - target sequence
            teacher_forcing_ratio: probability of using teacher forcing
        
        Returns:
            outputs: [batch_size, tgt_len, output_dim] - predicted sequences
        """
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.output_dim
        
        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)
        
        # Encode source sequence
        encoder_outputs, hidden, cell = self.encoder(src, src_lengths)
        
        # First input to decoder is SOS token
        input = tgt[:, 0]
        
        # Decode target sequence
        for t in range(1, tgt_len):
            # Pass through decoder
            output, hidden, cell = self.decoder(input, hidden, cell)
            
            # Store output
            outputs[:, t, :] = output
            
            # Teacher forcing: use ground truth or prediction
            teacher_force = random.random() < teacher_forcing_ratio
            
            # Get highest probability token
            top1 = output.argmax(1)
            
            # Use ground truth if teacher forcing, else use prediction
            input = tgt[:, t] if teacher_force else top1
        
        return outputs

### 3.4 Initialize Model

In [ ]:
# Model hyperparameters
INPUT_DIM = len(en_vocab)
OUTPUT_DIM = len(fr_vocab)
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5

# Initialize encoder and decoder
encoder = Encoder(INPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
decoder = Decoder(OUTPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)

# Initialize Seq2Seq model
model = Seq2Seq(encoder, decoder, device).to(device)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

# Initialize weights
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)
print('Model weights initialized')

## 4. Training

### 4.1 Training Configuration

In [ ]:
# Training hyperparameters
N_EPOCHS = 15
LEARNING_RATE = 0.001
CLIP = 1
TEACHER_FORCING_RATIO = 0.5

# Loss function (ignore padding)
criterion = nn.CrossEntropyLoss(ignore_index=fr_vocab.PAD_IDX)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler (FIXED: removed 'verbose' parameter for newer PyTorch versions)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

print('Training configuration:')
print(f'Epochs: {N_EPOCHS}')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Teacher forcing ratio: {TEACHER_FORCING_RATIO}')
print(f'Gradient clipping: {CLIP}')

### 4.2 Training and Evaluation Functions

In [ ]:
def train_epoch(model, iterator, optimizer, criterion, clip):
    """Train model for one epoch"""
    model.train()
    epoch_loss = 0
    
    for i, (src, tgt, src_lengths, tgt_lengths) in enumerate(iterator):
        src = src.to(device)
        tgt = tgt.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        output = model(src, src_lengths, tgt, TEACHER_FORCING_RATIO)
        
        # Reshape for loss calculation
        # output: [batch_size, tgt_len, output_dim]
        # tgt: [batch_size, tgt_len]
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)  # Skip first token (SOS)
        tgt = tgt[:, 1:].reshape(-1)  # Skip first token (SOS)
        
        # Calculate loss
        loss = criterion(output, tgt)
        
        # Backward pass
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        # Update weights
        optimizer.step()
        
        epoch_loss += loss.item()
    
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    """Evaluate model on validation/test set"""
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for i, (src, tgt, src_lengths, tgt_lengths) in enumerate(iterator):
            src = src.to(device)
            tgt = tgt.to(device)
            
            # Forward pass (no teacher forcing)
            output = model(src, src_lengths, tgt, 0)  # Turn off teacher forcing
            
            # Reshape for loss calculation
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            tgt = tgt[:, 1:].reshape(-1)
            
            # Calculate loss
            loss = criterion(output, tgt)
            
            epoch_loss += loss.item()
    
    return epoch_loss / len(iterator)

def epoch_time(start_time, end_time):
    """Calculate elapsed time"""
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

### 4.3 Training Loop with Early Stopping

In [ ]:
# Training loop
best_valid_loss = float('inf')
patience = 3
patience_counter = 0

train_losses = []
val_losses = []

print('Starting training...\n')

for epoch in range(N_EPOCHS):
    start_time = time.time()
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP)
    
    # Evaluate
    valid_loss = evaluate(model, val_loader, criterion)
    
    end_time = time.time()
    
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    
    # Store losses
    train_losses.append(train_loss)
    val_losses.append(valid_loss)
    
    # Early stopping check
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
        print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
        print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')
        print(f'\t*** Best model saved! ***\n')
    else:
        patience_counter += 1
        print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
        print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
        print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')
        print(f'\tPatience: {patience_counter}/{patience}\n')
    
    # Update learning rate
    scheduler.step(valid_loss)
    
    # Early stopping
    if patience_counter >= patience:
        print(f'Early stopping triggered after {epoch+1} epochs')
        break

print('Training completed!')

### 4.4 Plot Training and Validation Loss

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss over Epochs')
plt.legend()
plt.grid(True)
plt.savefig('loss_chart.png', dpi=300, bbox_inches='tight')
plt.show()

print('Loss chart saved as loss_chart.png')

## 5. Evaluation

### 5.1 Load Best Model

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pth'))
print('Best model loaded!')

# Evaluate on test set
test_loss = evaluate(model, test_loader, criterion)
print(f'Test Loss: {test_loss:.3f} | Test PPL: {math.exp(test_loss):7.3f}')

### 5.2 Implement Translation Function

In [ ]:
def translate(sentence, model, en_vocab, fr_vocab, en_tokenizer, device, max_len=50):
    """
    Translate an English sentence to French
    
    Args:
        sentence: English sentence (string)
        model: trained Seq2Seq model
        en_vocab: English vocabulary
        fr_vocab: French vocabulary
        en_tokenizer: English tokenizer
        device: torch device
        max_len: maximum length of translation
    
    Returns:
        translation: French sentence (string)
    """
    model.eval()
    
    with torch.no_grad():
        # Tokenize English sentence
        tokens = en_tokenizer(sentence)
        tokens = tokens[:max_len]
        tokens.append(en_vocab.EOS_TOKEN)
        
        # Convert to indices
        indices = en_vocab.encode(tokens)
        
        # Convert to tensor
        src_tensor = torch.LongTensor(indices).unsqueeze(0).to(device)
        src_lengths = torch.LongTensor([len(indices)])
        
        # Encode source sentence
        encoder_outputs, hidden, cell = model.encoder(src_tensor, src_lengths)
        
        # Start with SOS token
        input_token = torch.LongTensor([fr_vocab.SOS_IDX]).to(device)
        
        # Store translation
        translation_indices = []
        
        # Decode
        for _ in range(max_len):
            # Pass through decoder
            output, hidden, cell = model.decoder(input_token, hidden, cell)
            
            # Get predicted token
            pred_token = output.argmax(1).item()
            
            # Stop if EOS token
            if pred_token == fr_vocab.EOS_IDX:
                break
            
            # Add to translation
            translation_indices.append(pred_token)
            
            # Next input
            input_token = torch.LongTensor([pred_token]).to(device)
        
        # Convert indices to tokens
        translation_tokens = fr_vocab.decode(translation_indices)
        
        # Join tokens to form sentence
        translation = ' '.join(translation_tokens)
        
        return translation

# Test translation function
test_sentence = "A man in a blue shirt is standing on a ladder."
translation = translate(test_sentence, model, en_vocab, fr_vocab, tokenize_en, device)
print(f'Source: {test_sentence}')
print(f'Translation: {translation}')

### 5.3 Calculate BLEU Score

In [ ]:
import nltk
nltk.download('punkt', quiet=True)

def calculate_bleu(model, dataset, en_vocab, fr_vocab, en_tokenizer, fr_tokenizer, device, max_samples=None):
    """
    Calculate BLEU score on a dataset
    
    Args:
        model: trained Seq2Seq model
        dataset: dataset to evaluate
        en_vocab: English vocabulary
        fr_vocab: French vocabulary
        en_tokenizer: English tokenizer
        fr_tokenizer: French tokenizer
        device: torch device
        max_samples: maximum number of samples to evaluate (None for all)
    
    Returns:
        bleu_score: average BLEU score
    """
    model.eval()
    
    references = []
    hypotheses = []
    
    num_samples = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    
    print(f'Calculating BLEU score on {num_samples} samples...')
    
    for i in range(num_samples):
        en_sentence, fr_sentence = dataset.data[i]
        
        # Get reference translation
        reference = fr_tokenizer(fr_sentence)
        references.append([reference])
        
        # Get model translation
        translation = translate(en_sentence, model, en_vocab, fr_vocab, en_tokenizer, device)
        hypothesis = translation.split()
        hypotheses.append(hypothesis)
        
        if (i + 1) % 100 == 0:
            print(f'Processed {i + 1}/{num_samples} samples')
    
    # Calculate corpus BLEU score
    bleu_score = corpus_bleu(references, hypotheses)
    
    return bleu_score

# Calculate BLEU score on test set
bleu_score = calculate_bleu(model, test_dataset, en_vocab, fr_vocab, tokenize_en, tokenize_fr, device)
print(f'\nTest BLEU score: {bleu_score * 100:.2f}')

## 6. Error Analysis

### 6.1 Translation Examples

In [ ]:
# Select 5 examples from test set for error analysis
example_indices = [0, 10, 50, 100, 200]

print('Translation Examples and Error Analysis\n')
print('=' * 100)

for idx in example_indices:
    en_sentence, fr_sentence = test_data[idx]
    
    # Get model translation
    translation = translate(en_sentence, model, en_vocab, fr_vocab, tokenize_en, device)
    
    # Calculate sentence BLEU
    reference = [tokenize_fr(fr_sentence)]
    hypothesis = translation.split()
    bleu = sentence_bleu(reference, hypothesis) * 100
    
    print(f'\nExample {idx + 1}:')
    print(f'Source (EN):     {en_sentence}')
    print(f'Reference (FR):  {fr_sentence}')
    print(f'Prediction (FR): {translation}')
    print(f'BLEU Score:      {bleu:.2f}')
    print('-' * 100)

print('\n' + '=' * 100)

### 6.2 Error Analysis and Improvement Suggestions

In [ ]:
print("""\n## Error Analysis\n\n### Common Error Types:\n\n1. **Rare Words (Out-of-Vocabulary)**\n   - Problem: Words not in the 10,000 word vocabulary are replaced with <unk>\n   - Example: Proper nouns, technical terms, or rare words\n   - Impact: Loss of semantic information\n\n2. **Long Sentences**\n   - Problem: Fixed context vector loses information for long sequences\n   - Example: Sentences > 20 words may lose details\n   - Impact: Missing or incorrect words in translation\n\n3. **Grammar and Word Order**\n   - Problem: Model may generate grammatically incorrect French\n   - Example: Incorrect verb conjugation, gender agreement, or word order\n   - Impact: Unnatural or incorrect translations\n\n4. **Missing or Extra Words**\n   - Problem: Decoder may skip words or add unnecessary ones\n   - Example: Missing adjectives or adding repeated words\n   - Impact: Incomplete or verbose translations\n\n### Suggested Improvements:\n\n1. **Attention Mechanism**\n   - Add Bahdanau or Luong attention\n   - Allows decoder to focus on relevant encoder states\n   - Expected improvement: +5-10 BLEU points\n\n2. **Beam Search**\n   - Replace greedy decoding with beam search (beam size 3-5)\n   - Explores multiple translation paths\n   - Expected improvement: +2-5 BLEU points\n\n3. **Subword Tokenization (BPE)**\n   - Use Byte Pair Encoding instead of word-level tokens\n   - Handles rare words and morphology better\n   - Expected improvement: +3-7 BLEU points\n\n4. **Larger Dataset**\n   - Train on WMT 2014 (~36M pairs vs 29K)\n   - More diverse examples\n   - Expected improvement: +10-15 BLEU points\n\n5. **Model Architecture Improvements**\n   - Increase LSTM layers (3-4 layers)\n   - Increase hidden size (1024)\n   - Add bidirectional encoder\n   - Expected improvement: +3-5 BLEU points\n""")

## Summary

This notebook implements an Encoder-Decoder LSTM model for English-French machine translation from scratch using PyTorch.

### Key Components:

1. **Data Preparation**
   - Multi30K dataset (29,000 training pairs)
   - SpaCy tokenization
   - Vocabulary building (10,000 words per language)
   - Custom DataLoader with padding and packing

2. **Model Architecture**
   - Encoder: Embedding + 2-layer LSTM
   - Decoder: Embedding + 2-layer LSTM + Linear
   - Fixed context vector (no attention)
   - Teacher forcing ratio: 0.5

3. **Training**
   - Adam optimizer (lr=0.001)
   - Cross-entropy loss (ignore padding)
   - Early stopping (patience=3)
   - Learning rate scheduling

4. **Evaluation**
   - BLEU score on test set
   - Translation examples
   - Error analysis

### Files Generated:
- `best_model.pth`: Best model checkpoint
- `loss_chart.png`: Training/validation loss chart